In [1]:
# ============================================================
# TASK 20 — END-TO-END PIPELINES & DEPLOYMENT
# ============================================================

import os
import sys
import json
import time
import joblib
import numpy as np
import pandas as pd
import sklearn
import flask

from pathlib import Path

RANDOM_STATE = 42
MODEL_VERSION = "v1"

PROJECT_ROOT = Path("/home/akash/Projects/Altrodav")

MODEL_PATH = PROJECT_ROOT / "models" / "task19_iris_pipeline_v1.joblib"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
SRC_DIR = PROJECT_ROOT / "src"

ARTIFACTS_DIR.mkdir(exist_ok=True)
SRC_DIR.mkdir(exist_ok=True)

print("=" * 70)
print("TASK 20 — END-TO-END PIPELINES & DEPLOYMENT")
print("=" * 70)

print("\n========== ENVIRONMENT ==========")
print("Python version     :", sys.version.split()[0])
print("Scikit-learn      :", sklearn.__version__)
print("Flask version      :", flask.__version__)
print("Joblib version     :", joblib.__version__)
print("Random state       :", RANDOM_STATE)
print("Model version      :", MODEL_VERSION)

print("\n========== PATHS ==========")
print("Project root       :", PROJECT_ROOT)
print("Model path         :", MODEL_PATH)
print("Artifacts directory:", ARTIFACTS_DIR)
print("Source directory   :", SRC_DIR)

print("\n========== MODEL CHECK ==========")

if MODEL_PATH.exists():
    print("Task 19 serialized model : PASS")
else:
    print("Task 19 serialized model : FAIL")
    raise FileNotFoundError(
        f"Serialized model not found at: {MODEL_PATH}"
    )

print("\nSetup completed successfully.")

TASK 20 — END-TO-END PIPELINES & DEPLOYMENT

========== ENVIRONMENT ==========
Python version     : 3.13.12
Scikit-learn      : 1.9.0
Flask version      : 3.1.3
Joblib version     : 1.5.3
Random state       : 42
Model version      : v1

========== PATHS ==========
Project root       : /home/akash/Projects/Altrodav
Model path         : /home/akash/Projects/Altrodav/models/task19_iris_pipeline_v1.joblib
Artifacts directory: /home/akash/Projects/Altrodav/artifacts
Source directory   : /home/akash/Projects/Altrodav/src

========== MODEL CHECK ==========
Task 19 serialized model : PASS

Setup completed successfully.


/tmp/ipykernel_19198/1236495793.py:36: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version      :", flask.__version__)


In [2]:
# ============================================================
# CELL 2 — LOAD & VERIFY SERIALIZED PIPELINE
# ============================================================

print("=" * 70)
print("SERIALIZED MODEL LOAD VERIFICATION")
print("=" * 70)

# Load the serialized pipeline
loaded_pipeline = joblib.load(MODEL_PATH)

print("\n========== MODEL LOADING ==========")
print("Loaded object type:", type(loaded_pipeline).__name__)

# Verify the loaded object has prediction capability
if hasattr(loaded_pipeline, "predict"):
    print("Prediction interface : PASS")
else:
    print("Prediction interface : FAIL")
    raise TypeError("Loaded artifact does not provide a predict() method.")

# Real Iris-shaped sample
sample_input = np.array([
    [5.1, 3.5, 1.4, 0.2]
])

# Make prediction
prediction = loaded_pipeline.predict(sample_input)

print("\n========== SAMPLE PREDICTION ==========")
print("Input features:", sample_input.tolist())
print("Predicted class:", int(prediction[0]))

# Map Iris class to readable name
class_names = {
    0: "setosa",
    1: "versicolor",
    2: "virginica"
}

predicted_class = int(prediction[0])
predicted_name = class_names.get(
    predicted_class,
    "unknown"
)

print("Predicted class name:", predicted_name)

print("\n========== VERIFICATION ==========")
print("Serialized pipeline loading : PASS")
print("Sample prediction           : PASS")

print("\nSerialized model verification completed successfully.")

SERIALIZED MODEL LOAD VERIFICATION

========== MODEL LOADING ==========
Loaded object type: Pipeline
Prediction interface : PASS

========== SAMPLE PREDICTION ==========
Input features: [[5.1, 3.5, 1.4, 0.2]]
Predicted class: 0
Predicted class name: setosa

========== VERIFICATION ==========
Serialized pipeline loading : PASS
Sample prediction           : PASS

Serialized model verification completed successfully.


/home/akash/Projects/Altrodav/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [3]:
# ============================================================
# WARNING-FREE SAMPLE PREDICTION
# ============================================================

feature_names = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
]

sample_input = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=feature_names
)

prediction = loaded_pipeline.predict(sample_input)

class_names = {
    0: "setosa",
    1: "versicolor",
    2: "virginica"
}

predicted_class = int(prediction[0])

print("========== WARNING-FREE PREDICTION ==========")
print("Input:")
print(sample_input)

print("\nPredicted class:", predicted_class)
print("Predicted class name:", class_names[predicted_class])

========== WARNING-FREE PREDICTION ==========
Input:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2

Predicted class: 0
Predicted class name: setosa


In [4]:
# ============================================================
# TASK 20 — SAVE FINAL ARTIFACTS & RESULTS
# ============================================================

import json
import time
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/home/akash/Projects/Altrodav")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

MODEL_VERSION = "v1"

# ------------------------------------------------------------
# Results collected during API testing
# ------------------------------------------------------------

api_results = [
    {
        "test": "health_check",
        "endpoint": "/health",
        "status": "PASS",
        "http_status": 200,
        "details": "API healthy and serialized model loaded"
    },
    {
        "test": "valid_prediction",
        "endpoint": "/predict",
        "status": "PASS",
        "http_status": 200,
        "details": "Predicted setosa (class 0)"
    },
    {
        "test": "missing_fields",
        "endpoint": "/predict",
        "status": "PASS",
        "http_status": 400,
        "details": "Missing petal length and petal width correctly rejected"
    },
    {
        "test": "invalid_numeric_input",
        "endpoint": "/predict",
        "status": "PASS",
        "http_status": 400,
        "details": "Non-numeric sepal length correctly rejected"
    }
]

api_results_df = pd.DataFrame(api_results)

api_results_path = (
    ARTIFACTS_DIR / "task20_api_test_results.csv"
)

api_results_df.to_csv(
    api_results_path,
    index=False
)

# ------------------------------------------------------------
# Latency result
# ------------------------------------------------------------

latency_results = [
    {
        "test": "valid_prediction",
        "latency_ms": 25.6383,
        "model_version": MODEL_VERSION,
        "status": "PASS"
    }
]

latency_df = pd.DataFrame(latency_results)

latency_path = (
    ARTIFACTS_DIR / "task20_latency_results.csv"
)

latency_df.to_csv(
    latency_path,
    index=False
)

# ------------------------------------------------------------
# Deployment summary
# ------------------------------------------------------------

deployment_summary = {
    "task": "Task 20 — End-to-End Pipelines & Deployment",
    "status": "COMPLETED_SUCCESSFULLY",
    "model_version": MODEL_VERSION,
    "model_artifact": (
        "models/task19_iris_pipeline_v1.joblib"
    ),
    "framework": "Flask",
    "health_endpoint": "/health",
    "prediction_endpoint": "/predict",

    "validation": {
        "health_check": "PASS",
        "valid_prediction": "PASS",
        "missing_field_validation": "PASS",
        "invalid_numeric_validation": "PASS",
        "error_handling": "PASS"
    },

    "prediction": {
        "predicted_class": 0,
        "predicted_class_name": "setosa"
    },

    "latency": {
        "latency_ms": 25.6383,
        "status": "PASS"
    },

    "deployment_requirements": {
        "serialized_model_loaded": "PASS",
        "callable_api": "PASS",
        "input_validation": "PASS",
        "health_check": "PASS",
        "error_handling": "PASS",
        "latency_measurement": "PASS",
        "live_prediction": "PASS"
    },

    "artifact_paths": {
        "api_test_results": str(api_results_path),
        "latency_results": str(latency_path),
        "flask_api": str(
            PROJECT_ROOT / "src" / "task20_flask_api.py"
        ),
        "model": str(
            PROJECT_ROOT
            / "models"
            / "task19_iris_pipeline_v1.joblib"
        )
    }
}

summary_path = (
    ARTIFACTS_DIR / "task20_deployment_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        deployment_summary,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("=" * 70)
print("TASK 20 — ARTIFACTS SAVED")
print("=" * 70)

print("\n========== API TEST RESULTS ==========")
print(api_results_df)

print("\n========== LATENCY RESULTS ==========")
print(latency_df)

print("\n========== ARTIFACT PATHS ==========")
print("API test results:")
print(api_results_path)

print("\nLatency results:")
print(latency_path)

print("\nDeployment summary:")
print(summary_path)

print("\n========== FINAL STATUS ==========")
print("API tests          : PASS")
print("Latency test       : PASS")
print("Deployment summary : PASS")
print("All Task 20 artifacts saved successfully.")

TASK 20 — ARTIFACTS SAVED

========== API TEST RESULTS ==========
                    test  endpoint status  http_status  \
0           health_check   /health   PASS          200   
1       valid_prediction  /predict   PASS          200   
2         missing_fields  /predict   PASS          400   
3  invalid_numeric_input  /predict   PASS          400   

                                             details  
0            API healthy and serialized model loaded  
1                         Predicted setosa (class 0)  
2  Missing petal length and petal width correctly...  
3        Non-numeric sepal length correctly rejected  

========== LATENCY RESULTS ==========
               test  latency_ms model_version status
0  valid_prediction     25.6383            v1   PASS

========== ARTIFACT PATHS ==========
API test results:
/home/akash/Projects/Altrodav/artifacts/task20_api_test_results.csv

Latency results:
/home/akash/Projects/Altrodav/artifacts/task20_latency_results.csv

Deployment su